In [1]:
from utils import *
from time import time

In [2]:
r = 3
g = 3
d = 1 # no tocar
X = Curve("X", g)
J = Jacobian(X).to_lambda()

In [3]:
obj = get_motive_chow(X, r, d)
print(obj)

calculating from scratch...
h3_X**2*L**11 + h3_X**2*L**10 + 3*h3_X**2*L**9 + 3*h3_X**2*L**8 + 4*h3_X**2*L**7 + 3*h3_X**2*L**6 + 3*h3_X**2*L**5 + h3_X**2*L**4 + h3_X**2*L**3 + h3_X*L**14 + 2*h3_X*L**13 + 4*h3_X*L**12 + 7*h3_X*L**11 + 10*h3_X*L**10 + h3_X*L**9*λ2(h3_X) + 12*h3_X*L**9 + 2*h3_X*L**8*λ2(h3_X) + 14*h3_X*L**8 + 3*h3_X*L**7*λ2(h3_X) + h3_X*L**7*λ3(h3_X) + 14*h3_X*L**7 + 3*h3_X*L**6*λ2(h3_X) + 12*h3_X*L**6 + 2*h3_X*L**5*λ2(h3_X) + h3_X*L**5*λ3(h3_X) + 10*h3_X*L**5 + h3_X*L**4*λ2(h3_X) + 7*h3_X*L**4 + 4*h3_X*L**3 + 2*h3_X*L**2 + h3_X*L + L**16 + L**15 + 3*L**14 + 4*L**13 + L**12*λ2(h3_X) + 7*L**12 + L**11*λ2(h3_X) + 8*L**11 + 4*L**10*λ2(h3_X) + L**10*λ3(h3_X) + 11*L**10 + 5*L**9*λ2(h3_X) + L**9*λ3(h3_X) + 11*L**9 + 7*L**8*λ2(h3_X) + 2*L**8*λ3(h3_X) + 13*L**8 + 6*L**7*λ2(h3_X) + 2*L**7*λ3(h3_X) + 11*L**7 + L**6*λ2(h3_X)**2 + 7*L**6*λ2(h3_X) + 2*L**6*λ3(h3_X) + 11*L**6 + 5*L**5*λ2(h3_X) + 2*L**5*λ3(h3_X) + 8*L**5 + 4*L**4*λ2(h3_X) + L**4*λ3(h3_X) + 7*L**4 + L**3*λ2(h3_X) + L**3*λ3

In [4]:
expr = symbolize_chow(obj, X)
sorted(expr.free_symbols-{L}, key=lambda x: (len(x.name), x.name), reverse=True)

[λ3(h1_X), λ2(h1_X), λ1(h1_X)]

In [5]:
monoms = get_small_monomials(X, r)
max_dim = (r**2-1)*(g-1)
coefs = get_coefficients(max_dim)
monoms[4] = [(sym_lambda(X, 3)*sym_lambda(X, 1), X.get_lambda_var(3)*X.get_lambda_var(1))]

In [6]:
def find_motives_dfs(obj: LambdaRingExpr, X: Curve, coefs: list[LambdaRingExpr], monoms, n_rounds: int, max_dim: int=-1) -> list[LambdaRingExpr]:
    # CENTRARSE EN ESTO, EL PROBLEMA ES LA MEMORIA
    """
    returns, if found, a motivic decompostion given monomials (high degree) and coefficients
    """
    max_degree = max(monoms.keys())
    min_degree = min(monoms.keys())
    n_monoms = len(monoms[max_degree])
    state0 = {
        "degree": max_degree,
        "monom_idx": 0,
        "round": 0,
    }
    frontier = [(obj, 0, state0)]
    while frontier:
        remaining, big_part, state = frontier.pop()
        if any(term.could_extract_minus_sign() for term in remaining.as_ordered_terms()):
            continue
        candidate = find_motive_low(remaining, X)
        if candidate is not None:
            print(big_part)
            print(remaining)
            return {candidate + big_part}
        monom_X, monom_H = monoms[state["degree"]][state["monom_idx"]]
        highest_coef_degree = len(coefs) if max_dim == -1 else max(max_dim-state["degree"]+1, 0)
        new_state = {
            "degree": state["degree"],
            "monom_idx": state["monom_idx"],
            "round": state["round"] + 1,       
        }
        if new_state["round"] == n_rounds:
            new_state["round"] = 0
            new_state["monom_idx"] += 1
        if new_state["monom_idx"] == n_monoms:
            new_state["monom_idx"] = 0
            new_state["degree"] -= 1 
        if new_state["degree"] < min_degree:
            continue
        n_monoms = len(monoms[new_state["degree"]])
        for coef in coefs[:highest_coef_degree]+[0]:
            m = (remaining - coef*monom_H).expand()
            node = (m, big_part+coef*monom_X, new_state)  
            if node not in frontier:
                frontier.append(node)
    return set()

In [7]:
def find_motives_dfs2(obj: LambdaRingExpr, X: Curve, coefs: list[LambdaRingExpr], monoms, n_rounds: int, max_dim: int=-1) -> set[LambdaRingExpr]: 
    """
    returns all possible motivic decompositions given monomials (high degree) and coefficients
    """
    candidates = {(obj, 0)}
    for degree in sorted(monoms.keys(), reverse=True):
        highest_coef_degree = len(coefs) if max_dim == -1 else max(max_dim-degree+1, 0) # cut coefs (revisar?)
        for monom_X, monom_H in tqdm(monoms[degree]):
            for i in range(n_rounds):
                # print(f"{degree} degree monomial, {i+1} round, {len(candidates)} candidates")
                new_candidates = set()
                size_candidates = len(candidates)
                for candidate, big_part in candidates:
                    for coef in coefs[:highest_coef_degree]:   # cut coefs 
                        m = (candidate - coef*monom_H).expand()
                        if not any(term.could_extract_minus_sign() for term in m.as_ordered_terms()):
                            cand = find_motive_low(m, X)
                            if cand is not None:
                                return {big_part + monom_X*coef + cand}
                            new_candidates.add((m, big_part+coef*monom_X))    # revisar unicidad de motivos
                            size_candidates += 1
                candidates = candidates.union(new_candidates)
                print(len(candidates))
                print(size_candidates)
    return set()

In [17]:
t1 = time()
motives = find_motives_dfs2(obj, X, coefs, monoms, 6)
print(time()-t1)
motives

  0%|          | 0/1 [00:00<?, ?it/s]

3
3
4
7
4
8
4
8


100%|██████████| 1/1 [00:01<00:00,  1.01s/it]

4
8
4
8
1.0146424770355225


set()

In [9]:
motive = motives.pop()

KeyError: 'pop from an empty set'

In [ ]:
motive

L**15 + L**9*λ3(X) + L**7*λ1(X)*λ3(X) + L**7*λ3(X) + L**6*λ1(X)**3 + L**6*λ1(X)*λ3(X) + L**5*λ1(X)*λ2(X) + L**5*λ3(X) + L**4*λ1(X)*λ3(X) + L**3*λ3(X) + λ1(X)**2*(L**10 + L**9 + L**8 + L**5 + L**4 + L**3) + λ1(X)*(L**13 + L**12 + L**11 + L**9 + 2*L**8 + L**6 + L**5 + L**3 + L**2 + L) + λ2(X)*(L**11 + L**9 + L**7 + L**6 + L**4 + L**2) + 1

In [ ]:
save_expr(motive, X, f"{r}_{g}")